In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
fact_order=spark.read.format("delta").load("s3://retail-lakehouse-ashu/silver/website/orders/")
fact_order.display()
fact_order.createOrReplaceTempView("fact_order")
spark.sql("select distinct order_status from fact_order").display()
fact_order.select(col("order_status")).distinct().display()

In [0]:
fact_order_items=spark.read.format("delta").load("s3://retail-lakehouse-ashu/silver/website/order_items/")
fact_order_items.display()

In [0]:
dim_product=spark.read.format("delta").load("s3://retail-lakehouse-ashu/silver/erp/dim_product/")
dim_product.display()

In [0]:
orders=fact_order.alias("o")
order_items=fact_order_items.alias("oi")
joined_df=orders.join(order_items,on='order_id',how="inner").filter(col("order_status")=="DELIVERED").select(col("order_id"),col("oi.customer_id"),col("order_timestamp"),col("product_id"),col("product_name"),col("quantity"),col("price")).withColumn("order_date",to_date(col("order_timestamp")))
joined_df.display()

In [0]:
group_df=joined_df.groupby(col("order_date"),col("product_id"),col("product_name")).agg(sum(col("quantity")).alias("total_quantity"),sum(col("quantity")*col("price")).alias("gross_revenue"),countDistinct(col("order_id")).alias("total_orders"))
group_df.display()

In [0]:
dim_customer=spark.read.format("delta").load("s3://retail-lakehouse-ashu/silver/crm/dim_customer/")
dim_customer.display()

In [0]:
customer_join=joined_df.join(dim_customer,on='customer_id',how="inner").select(col("customer_id"),col("order_id"),col("quantity"),col("price"),col("first_name"),col("last_name"))
customer_join.display()

In [0]:
group_cust=customer_join.groupby(col("customer_id"),col("first_name"),col("last_name")).agg(sum(col("quantity")).alias("total_quantity"),sum(col("quantity")*col("price")).alias("gross_revenue"),countDistinct(col("order_id")).alias("total_orders"))
customer_sales_summary=group_cust.withColumn("avg_price",col("gross_revenue")/col("total_orders"))
customer_sales_summary.display()

In [0]:
product_performance_summary=joined_df.groupby(col("product_id"),col("product_name")).agg(sum(col("quantity")).alias("total_quantity"),sum(col("quantity")*col("price")).alias("gross_revenue"),countDistinct(col("order_id")).alias("total_orders"))
product_performance_summary_final=product_performance_summary.withColumn("average_unit_price",col("gross_revenue")/col("total_quantity"))
product_performance_summary_final.display()